In [1]:
%pip install -q -U ir_measures sentence_transformers rank_bm25 PyStemmer ranknexus

In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
import hashlib

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers import CrossEncoder
from sentence_transformers import util

import ir_measures
from ir_measures import nDCG, RR, Recall

from ranknexus import fuse

In [3]:
dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic") # old ds without removed documents for full unique documents ds

traind_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['document_text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['query', 'query_type', 'difficulty', 'index'])

unique_docs_df = unique_docs_df.rename(columns={"document_text": "text", "document_link": "link", "document_title": "title"})

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.93MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  743kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4612 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1153 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'text', 'link'],
    num_rows: 1153
})

In [4]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [5]:
FT_RETRIEVER_MODEL_ID = "pameydorke/arcanumchat-retriever-bge-m3-v2"
FT_RERANKER_MODEL_ID = "pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-v2"

FT_HS_RERANKER_MODEL_ID = "pameydorke/arcanumchat-reranker-bge-reranker-v2-m3-hybrid-search"

ft_retriever_model = SentenceTransformer(FT_RETRIEVER_MODEL_ID)
ft_reranker_model = CrossEncoder(FT_RERANKER_MODEL_ID, num_labels=1)

ft_hs_reranker_model = CrossEncoder(FT_HS_RERANKER_MODEL_ID, num_labels=1)

ft_retriever_model.eval()
ft_reranker_model.eval()

ft_hs_reranker_model.eval()

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/42.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (22 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [6]:
queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["link"]
    doc_text = example["text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        doc_id = hashlib.sha1(doc_text[:200].encode("utf-8")).hexdigest()[:16]
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
vague_counter = 0
for idx, example in enumerate(test_ds):
    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)

Skipped: 0


In [7]:
def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [8]:
def per_query_metrics(model, reranker, queries, corpus, relevant_docs, k_values=(1, 3, 5, 10), k_retrieval=10):
    assert k_retrieval >= max(k_values)
    assert not model.training, "bi-encoder in train mode"

    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    sim = util.cos_sim(q_emb, c_emb)
    initial_ranked_indices = torch.argsort(sim, dim=1, descending=True)[:, :k_retrieval]
    initial_ranked_cids = [[cids[i] for i in row] for row in initial_ranked_indices.tolist()]

    reranked_cids = []
    for i, qid in enumerate(queries):
        query_text = queries[qid]
        candidate_cids = initial_ranked_cids[i]
        candidate_docs = [corpus[cid] for cid in candidate_cids]

        pairs = [(query_text, doc) for doc in candidate_docs]
        scores = reranker.predict(pairs, convert_to_numpy=True)

        assert scores.ndim == 1, f"predict returned {scores.shape}"

        order = np.argsort(scores)[::-1]
        reranked_cids.append([candidate_cids[j] for j in order])

    # Build qrels and run dictionaries for ir_measures
    qrels = {qid: {doc_id: 1 for doc_id in rel_set} for qid, rel_set in relevant_docs.items()}
    run = {
        qid: {doc_id: float(len(reranked_cids[i]) - rank) for rank, doc_id in enumerate(reranked_cids[i])}
        for i, qid in enumerate(qids)
    }

    measures = []
    for k in k_values:
        measures.extend([Recall @ k, nDCG @ k])
    measures.append(RR)  # RR gives reciprocal rank per query; averaging yields MRR

    results = {qid: {} for qid in qids}
    for metric in ir_measures.iter_calc(measures, qrels, run):
        qid = metric.query_id
        name = str(metric.measure)
        if name == "RR":
            name = "mrr"  # Per-query RR is the MRR contribution
        results[qid][name] = metric.value

    return results

In [9]:
import Stemmer
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_stemmer = Stemmer.Stemmer("english")

def tokenize(text: str):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return _stemmer.stemWords(tokens)

tokenized_corpus = [tokenize(doc) for doc in list(corpus.values())]
bm25 = BM25Okapi(tokenized_corpus)

In [11]:
def per_query_metrics_hs(
    model,
    reranker,
    queries,
    corpus,
    relevant_docs,
    k_values=(5, 10, 20),
    k_retrieval_dense=10,
    k_retrieval_bm25=10,
    rrf_k=60,
):
    assert (k_retrieval_dense + k_retrieval_bm25) >= max(k_values)

    assert not model.training, "bi-encoder in train mode"
    assert not reranker.training, "cross-encoder in train mode"

    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    # --- Dense retrieval ---
    sim = util.cos_sim(q_emb, c_emb)
    dense_topk = torch.argsort(sim, dim=1, descending=True)[:, :k_retrieval_dense]
    dense_ranked_cids = [[cids[i] for i in row] for row in dense_topk.tolist()]

    # --- BM25 retrieval ---
    bm25_ranked_cids = []
    for qid in qids:
        query_text = queries[qid]
        scores = bm25.get_scores(tokenize(query))
        top_idx = np.argsort(scores)[::-1][:k_retrieval_bm25]
        bm25_ranked_cids.append([cids[i] for i in top_idx])

    # --- Reciprocal Rank Fusion ---
    fused_ranked_cids = []
    for i in range(len(qids)):
        dense_ranking = [{"id": cid} for cid in dense_ranked_cids[i]]
        bm25_ranking = [{"id": cid} for cid in bm25_ranked_cids[i]]

        fused_results = fuse(
            [dense_ranking, bm25_ranking],
            method="rrf",
            k=rrf_k,
            top_k=None, # return the full fused list for reranking
        )

        fused_ranked_cids.append([res.document_id for res in fused_results])

    # --- Cross-encoder reranking over the fused candidate pool ---
    reranked_cids = []
    for i, qid in enumerate(qids):
        query_text = queries[qid]
        candidate_cids = fused_ranked_cids[i]
        candidate_docs = [corpus[cid] for cid in candidate_cids]

        pairs = [(query_text, doc) for doc in candidate_docs]
        scores = reranker.predict(pairs, convert_to_numpy=True)
        assert scores.ndim == 1, f"predict returned {scores.shape}"

        order = np.argsort(scores)[::-1]
        reranked_cids.append([candidate_cids[j] for j in order])

    # --- Build qrels and run dictionaries for ir_measures ---
    qrels = {qid: {doc_id: 1 for doc_id in rel_set} for qid, rel_set in relevant_docs.items()}
    run = {
        qid: {
            doc_id: float(len(reranked_cids[i]) - rank)
            for rank, doc_id in enumerate(reranked_cids[i])
        }
        for i, qid in enumerate(qids)
    }

    measures = []
    for k in k_values:
        measures.extend([Recall @ k, nDCG @ k])
    measures.append(RR)  # per-query RR -> averaged yields MRR

    results = {qid: {} for qid in qids}
    for metric in ir_measures.iter_calc(measures, qrels, run):
        qid = metric.query_id
        name = str(metric.measure)
        if name == "RR":
            name = "mrr"
        results[qid][name] = metric.value

    return results

In [12]:
metrics = ["R@5", "R@10", "nDCG@10"]

def print_metrics(base_per_q, hs_per_q):
    for metric in metrics:
        base = np.array([base_per_q[q][metric] for q in queries])
        hs   = np.array([hs_per_q[q][metric]   for q in queries])
        delta = hs - base

        print(metric)
        print("base mean:", base.mean())
        print("hs mean:  ", hs.mean())
        print("improved:", (delta > 0).sum())
        print("worse:   ", (delta < 0).sum())
        print("same:    ", (delta == 0).sum())

        rng = np.random.default_rng(0)
        n = len(delta)

        boot = np.array([
            delta[rng.integers(0, n, n)].mean()
            for _ in range(10_000)
        ])

        lo, hi = np.percentile(boot, [2.5, 97.5])
        # print(f"mean delta = {delta.mean():.4f}")
        print(f"95% CI     = [{lo:.4f}, {hi:.4f}]", "\n")

In [13]:
base_per_q_10 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

hs_per_q_10_10 = per_query_metrics_hs(ft_retriever_model, ft_hs_reranker_model, queries, corpus, relevant_docs, k_retrieval_dense=10, k_retrieval_bm25=10)

print_metrics(base_per_q_10, hs_per_q_10_10)

R@5
base mean: 0.8958333333333334
hs mean:   0.8958333333333334
improved: 1
worse:    1
same:     190
95% CI     = [-0.0156, 0.0156] 

R@10
base mean: 0.921875
hs mean:   0.9166666666666666
improved: 0
worse:    1
same:     191
95% CI     = [-0.0156, 0.0000] 

nDCG@10
base mean: 0.8487255110262363
hs mean:   0.8505210041745848
improved: 10
worse:    11
same:     171
95% CI     = [-0.0149, 0.0182] 



In [14]:
base_per_q_20 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

hs_per_q_20_20 = per_query_metrics_hs(ft_retriever_model, ft_hs_reranker_model, queries, corpus, relevant_docs, k_retrieval_dense=20, k_retrieval_bm25=20)

print_metrics(base_per_q_20, hs_per_q_20_20)

R@5
base mean: 0.921875
hs mean:   0.921875
improved: 4
worse:    4
same:     184
95% CI     = [-0.0260, 0.0312] 

R@10
base mean: 0.9583333333333334
hs mean:   0.9583333333333334
improved: 1
worse:    1
same:     190
95% CI     = [-0.0156, 0.0156] 

nDCG@10
base mean: 0.8705032704103971
hs mean:   0.8810948318859476
improved: 18
worse:    11
same:     163
95% CI     = [-0.0086, 0.0299] 



#### ft_reranker_model, rank_bm25.BM25Okapi and basic tokenizer(.lower().split())

In [ ]:
base_per_q_10 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=10)

hs_per_q_10_10 = per_query_metrics_hs(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval_dense=10, k_retrieval_bm25=10)

print_metrics(base_per_q_10, hs_per_q_10_10)

base mean: 0.8958333333333334
hs mean:   0.9166666666666666
improved: 4
worse:    0
same:     188
mean delta = 0.0208
95% CI     = [0.0052, 0.0417] 

base mean: 0.921875
hs mean:   0.921875
improved: 4
worse:    4
same:     184
mean delta = 0.0000
95% CI     = [-0.0312, 0.0260] 

base mean: 0.8487255110262363
hs mean:   0.8443905263570852
improved: 4
worse:    14
same:     174
mean delta = -0.0043
95% CI     = [-0.0223, 0.0162] 



In [ ]:
base_per_q_20 = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

hs_per_q_20_20 = per_query_metrics_hs(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval_dense=20, k_retrieval_bm25=20)

print_metrics(base_per_q_20, hs_per_q_20_20)

base mean: 0.921875
hs mean:   0.90625
improved: 0
worse:    3
same:     189
mean delta = -0.0156
95% CI     = [-0.0365, 0.0000] 

base mean: 0.9583333333333334
hs mean:   0.9427083333333334
improved: 1
worse:    4
same:     187
mean delta = -0.0156
95% CI     = [-0.0417, 0.0052] 

base mean: 0.8705032704103971
hs mean:   0.8605428516142165
improved: 1
worse:    13
same:     178
mean delta = -0.0100
95% CI     = [-0.0196, -0.0016] 



In [28]:
hs_per_q_20_10 = per_query_metrics_hs(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval_dense=20, k_retrieval_bm25=10)

print_metrics(base_per_q_20, hs_per_q_20_10)

base mean: 0.921875
hs mean:   0.90625
improved: 0
worse:    3
same:     189
mean delta = -0.0156
95% CI     = [-0.0365, 0.0000] 

base mean: 0.9583333333333334
hs mean:   0.9479166666666666
improved: 1
worse:    3
same:     188
mean delta = -0.0104
95% CI     = [-0.0312, 0.0104] 

base mean: 0.8705032704103971
hs mean:   0.8622345273403264
improved: 1
worse:    12
same:     179
mean delta = -0.0083
95% CI     = [-0.0173, -0.0005] 

